# Vector算子开发实战

实现Ascend C算子LogSigmoid，算子命名为LogSigmoidCustom，编写其kernel侧代码、host侧代码，并完成单算子调用测试。
相关算法：
  $$
  LogSigmoid(x) = \log(\frac{1}{1+\exp(-x)})
  $$

要求：

- 完成LogSigmoid算子kernel侧核函数实现相关代码补齐。

- 完成LogSigmoid算子host侧Tiling函数相关代码补齐。

- 要支持float/half/bfloat16类型输入输出。

- 支持多条调用case并运行成功：
    * case1：输入x的shape为(8,16,64)，类型为float32
    * case2：输入x的shape为(8,16,1743)，类型为float32
    * case3：输入x的shape为(4,2028)，类型为float16
    * case4：输入x的shape为(32,1001,7763)，类型为float16
    * case5：输入x的shape为(1,1024)，类型为bfloat16
    * case6：输入x的shape为(64,64,1024)，类型为bfloat16

请开始你的实践，体验完整开发过程。

---

### 环境准备：

In [1]:
!mkdir -p Sources/test

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")


🎉 Environment initialization process completed successfully!


算子工程创建工具msopgen准备：

In [2]:
# 下载工具仓代码
!git clone https://gitcode.com/Ascend/msopgen.git
#编译安装msopgen工具
!cd msopgen; python build.py;cd output;pip install mindstudio_opgen*.whl --force-reinstall;pip install mindstudio_opst*.whl --force-reinstall


Cloning into 'msopgen'...
remote: Enumerating objects: 1127, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 1127 (delta 165), reused 98 (delta 98), pack-reused 896 (from 1)
Receiving objects: 100% (1127/1127), 1.26 MiB | 5.35 MiB/s, done.
Resolving deltas: 100% (490/490), done.
2026-06-25 15:47:45,624 - INFO - === Download git submodules start ===
2026-06-25 15:47:45,624 - INFO - Fetching third-party submodules...
2026-06-25 15:47:45,624 - INFO - Run CMD: git submodule update --init --progress --depth=1 --jobs=4 --recursive --remote thirdparty/asc-tools
Submodule 'thirdparty/asc-tools' (https://gitcode.com/cann/asc-tools.git) registered for path 'thirdparty/asc-tools'
Cloning into '/workspace/notebook23/cann-learning-hub/tutorials/ascendc_operator_development/09_course_practice/msopgen/thirdparty/asc-tools'...
remote: Enumerating objects: 438, done.        
remote: Counting objects: 100% (438/438), done.        
re

--- 
### 算子原型json文件准备
该部分不需要修改，直接执行即可。

In [3]:
%%writefile Sources/test/log_sigmoid_custom.json
[{
    "op": "LogSigmoidCustom",
    "input_desc": [{
            "name": "x",
            "param_type": "required",
            "format": ["ND", "ND", "ND"],
            "type": ["float16", "float", "bf16"]
        }
    ],
    "output_desc": [{
        "name": "y",
        "param_type": "required",
        "format": ["ND", "ND", "ND"],
        "type": ["float16", "float", "bf16"]
    }]
}]

Writing Sources/test/log_sigmoid_custom.json


---
### 算子工程创建
该部分不需要修改，直接执行即可。

In [4]:
# 清除已有的custom_op目录
!rm -rf Sources/test/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/test/log_sigmoid_custom.json -c ai_core-ascend910b1 -lan cpp -out Sources/test/custom_op


                   >>>>>   MindStudio   <<<<<
    THE END-TO-END TOOLCHAIN TO UNLEASH HUAWEI ASCEND COMPUTE

2026-06-25 15:48:28 (443115) - [INFO] Start to generate AI Core operator files. 
2026-06-25 15:48:28 (443115) - [INFO] Start to parse the ir template:/workspace/notebook23/cann-learning-hub/tutorials/ascendc_operator_development/09_course_practice/Sources/test/log_sigmoid_custom.json 
2026-06-25 15:48:28 (443115) - [INFO] Start to parse the op: LogSigmoidCustom 
2026-06-25 15:48:28 (443115) - [INFO] Start to parse the input_desc: x 
2026-06-25 15:48:28 (443115) - [INFO] Start to parse the output_desc: y 
2026-06-25 15:48:28 (443115) - [WARNING] The "attr" value is invalid or no "attr" exists in the map. 
2026-06-25 15:48:28 (443115) - [INFO] Start to check the type and format between the inputs/outputs in IR template. 
2026-06-25 15:48:28 (443115) - [INFO] Start to generate a new project. 
2026-06-25 15:48:28 (443115) - [INFO] File /workspace/notebook23/cann-learning-hub/tutoria

---
### host侧代码修改
修改代码，完成Tiling实现：

In [16]:
%%writefile Sources/test/custom_op/op_host/log_sigmoid_custom.cpp
#include "../op_kernel/log_sigmoid_custom_tiling.h"
#include "register/op_def_registry.h"
#include "graph/utils/type_utils.h"
#include "tiling/platform/platform_ascendc.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
    auto ascendcPlatform = platform_ascendc::PlatformAscendC(context->GetPlatformInfo());
    auto coreNum = ascendcPlatform.GetCoreNum();

    uint32_t inputNum = context->GetInputShape(0)->GetStorageShape().GetShapeSize();
    uint32_t typeLength = 0;
    ge::TypeUtils::GetDataTypeLength(context->GetInputDesc(0)->GetDataType(), typeLength);
    uint32_t inputLength = inputNum * typeLength;
    const uint32_t BLOCK_SIZE = 32;

    uint32_t inputLengthAlgin32 = (((inputLength + BLOCK_SIZE - 1) / BLOCK_SIZE) * BLOCK_SIZE);
    coreNum = std::min(coreNum, inputLengthAlgin32 / BLOCK_SIZE);
    coreNum = std::max(coreNum, static_cast<uint32_t>(1));
    uint32_t everyCoreInputBlockNum = inputLengthAlgin32 / BLOCK_SIZE / coreNum;
    uint32_t tailBlockNum = (inputLengthAlgin32 / BLOCK_SIZE) % coreNum;
    context->SetBlockDim(coreNum);

    uint64_t ubSize;
    ascendcPlatform.GetCoreMemSize(platform_ascendc::CoreMemType::UB, ubSize);
    const uint32_t BUFFER_NUM = 2;
    // bf16 needs extra TBuf for float32 intermediate (float is 2x bf16 size)
    uint32_t ubDataNumber = (context->GetInputDesc(0)->GetDataType() == ge::DT_BF16) ? 3 : 2;
    uint32_t tileBlockNum = (ubSize / BLOCK_SIZE / BUFFER_NUM) / ubDataNumber;
    uint32_t tileDataNum = (tileBlockNum * BLOCK_SIZE) / typeLength;

    uint32_t smallCoreDataNum = everyCoreInputBlockNum * BLOCK_SIZE / typeLength;
    uint32_t smallTileNum = everyCoreInputBlockNum / tileBlockNum;
    uint32_t finalSmallTileNum = (everyCoreInputBlockNum % tileBlockNum) == 0 ? smallTileNum : smallTileNum + 1;
    uint32_t smallTailDataNum = smallCoreDataNum - (tileDataNum * smallTileNum);
    smallTailDataNum = smallTailDataNum == 0 ? tileDataNum : smallTailDataNum;

    everyCoreInputBlockNum += 1;
    uint32_t bigCoreDataNum = everyCoreInputBlockNum * BLOCK_SIZE / typeLength;
    uint32_t bigTileNum = everyCoreInputBlockNum / tileBlockNum;
    uint32_t finalBigTileNum = (everyCoreInputBlockNum % tileBlockNum) == 0 ? bigTileNum : bigTileNum + 1;
    uint32_t bigTailDataNum = bigCoreDataNum - tileDataNum * bigTileNum;
    bigTailDataNum = bigTailDataNum == 0 ? tileDataNum : bigTailDataNum;

    LogSigmoidCustomTilingData *tiling = context->GetTilingData<LogSigmoidCustomTilingData>();
    tiling->smallCoreDataNum = smallCoreDataNum;
    tiling->bigCoreDataNum = bigCoreDataNum;
    tiling->tileDataNum = tileDataNum;
    tiling->smallTailDataNum = smallTailDataNum;
    tiling->bigTailDataNum = bigTailDataNum;
    tiling->finalSmallTileNum = finalSmallTileNum;
    tiling->finalBigTileNum = finalBigTileNum;
    tiling->tailBlockNum = tailBlockNum;

    size_t *currentWorkspace = context->GetWorkspaceSizes(1);
    currentWorkspace[0] = 0;
    return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}


namespace ops {
class LogSigmoidCustom : public OpDef {
public:
    explicit LogSigmoidCustom(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT, ge::DT_BF16})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT, ge::DT_BF16})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);

        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");

    }
};

OP_ADD(LogSigmoidCustom);
}

Overwriting Sources/test/custom_op/op_host/log_sigmoid_custom.cpp


---
### Tiling结构体定义修改
修改代码，完成Tiling结构体定义：

In [17]:
%%writefile  Sources/test/custom_op/op_kernel/log_sigmoid_custom_tiling.h
#ifndef LOG_SIGMOID_CUSTOM_TILING_H
#define LOG_SIGMOID_CUSTOM_TILING_H
#include <cstdint>

struct LogSigmoidCustomTilingData {
    uint32_t smallCoreDataNum;
    uint32_t bigCoreDataNum;
    uint32_t finalBigTileNum;
    uint32_t finalSmallTileNum;
    uint32_t tileDataNum;
    uint32_t smallTailDataNum;
    uint32_t bigTailDataNum;
    uint32_t tailBlockNum;
};

#endif // LOG_SIGMOID_CUSTOM_TILING_H

Overwriting Sources/test/custom_op/op_kernel/log_sigmoid_custom_tiling.h


---
### Kernel实现修改
修改代码，完成Kernel实现：

In [18]:
%%writefile  Sources/test/custom_op/op_kernel/log_sigmoid_custom.cpp
#include "kernel_operator.h"
#include "log_sigmoid_custom_tiling.h"

constexpr int32_t BUFFER_NUM = 2;

template<typename TYPE_X, typename TYPE_Y>
class KernelLogSigmoid {
public:
    __aicore__ inline KernelLogSigmoid() {}

    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, uint32_t smallCoreDataNum,
                                uint32_t bigCoreDataNum, uint32_t finalBigTileNum,
                                uint32_t finalSmallTileNum, uint32_t tileDataNum,
                                uint32_t smallTailDataNum, uint32_t bigTailDataNum,
                                uint32_t tailBlockNum)
    {
        uint32_t coreIdx = AscendC::GetBlockIdx();
        uint32_t globalBufferIndex = bigCoreDataNum * coreIdx;
        this->tileDataNum = tileDataNum;

        if (coreIdx < tailBlockNum) {
            this->coreDataNum = bigCoreDataNum;
            this->tileNum = finalBigTileNum;
            this->tailDataNum = bigTailDataNum;
        } else {
            this->coreDataNum = smallCoreDataNum;
            this->tileNum = finalSmallTileNum;
            this->tailDataNum = smallTailDataNum;
            globalBufferIndex -= (bigCoreDataNum - smallCoreDataNum) * (coreIdx - tailBlockNum);
        }

        xGm.SetGlobalBuffer((__gm__ TYPE_X*)x + globalBufferIndex, this->coreDataNum);
        yGm.SetGlobalBuffer((__gm__ TYPE_Y*)y + globalBufferIndex, this->coreDataNum);

        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileDataNum * sizeof(TYPE_X));
        pipe.InitBuffer(outQueueY, BUFFER_NUM, this->tileDataNum * sizeof(TYPE_Y));

        if constexpr (std::is_same_v<TYPE_X, bfloat16_t>) {
            pipe.InitBuffer(tmpBuf, this->tileDataNum * sizeof(float));
        }
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum;
        this->processDataNum = this->tileDataNum;
        for (int32_t i = 0; i < loopCount; i++) {
            if (i == this->tileNum - 1) {
                this->processDataNum = this->tailDataNum;
            }
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<TYPE_X> xLocal = inQueueX.AllocTensor<TYPE_X>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileDataNum], this->processDataNum);
        inQueueX.EnQue(xLocal);
    }

    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<TYPE_X> xLocal = inQueueX.DeQue<TYPE_X>();
        AscendC::LocalTensor<TYPE_Y> yLocal = outQueueY.AllocTensor<TYPE_Y>();

        if constexpr (std::is_same_v<TYPE_X, bfloat16_t>) {
            AscendC::LocalTensor<float> tmpLocal = tmpBuf.Get<float>();
            AscendC::Cast(tmpLocal, xLocal, AscendC::RoundMode::CAST_NONE, this->processDataNum);
            AscendC::Sigmoid(tmpLocal, tmpLocal, this->processDataNum);
            AscendC::Ln(tmpLocal, tmpLocal, this->processDataNum);
            AscendC::Cast(yLocal, tmpLocal, AscendC::RoundMode::CAST_RINT, this->processDataNum);
        } else {
            AscendC::Sigmoid(yLocal, xLocal, this->processDataNum);
            AscendC::Ln(yLocal, yLocal, this->processDataNum);
        }

        outQueueY.EnQue<TYPE_Y>(yLocal);
        inQueueX.FreeTensor(xLocal);
    }

    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<TYPE_Y> yLocal = outQueueY.DeQue<TYPE_Y>();
        AscendC::DataCopy(yGm[progress * this->tileDataNum], yLocal, this->processDataNum);
        outQueueY.FreeTensor(yLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueY;
    AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
    AscendC::GlobalTensor<TYPE_X> xGm;
    AscendC::GlobalTensor<TYPE_Y> yGm;
    uint32_t coreDataNum;
    uint32_t tileNum;
    uint32_t tileDataNum;
    uint32_t tailDataNum;
    uint32_t processDataNum;
};

extern "C" __global__ __aicore__ void log_sigmoid_custom(GM_ADDR x, GM_ADDR y, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(LogSigmoidCustomTilingData);
    GET_TILING_DATA(tilingData, tiling);
    KernelLogSigmoid<DTYPE_X, DTYPE_Y> op;
    op.Init(x, y, tilingData.smallCoreDataNum,
            tilingData.bigCoreDataNum, tilingData.finalBigTileNum,
            tilingData.finalSmallTileNum, tilingData.tileDataNum,
            tilingData.smallTailDataNum, tilingData.bigTailDataNum,
            tilingData.tailBlockNum);
    op.Process();
}

Overwriting Sources/test/custom_op/op_kernel/log_sigmoid_custom.cpp


修改完成后，部署算子运行测试，该部分不需要修改，直接执行即可。

In [19]:
# 编译部署修改后的算子
!cd Sources/test/custom_op;bash build.sh;./build_out/custom_opp*.run --install-path=${HOME}/

using ASCEND_HOME_PATH: /usr/local/Ascend/cann-8.5.0
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- set package config: INSTALL_PATH;/workspace/notebook23/cann-learning-hub/tutorials/ascendc_operator_development/09_course_practice/Sources/test/custom_op/build_out/
INFO target name: cust_tf_parsers
-- Opbuild generating sources
-- Opbuild generating sources - done
INFO target name: cust_optiling
INFO target name: cust_opapi
INFO target name: cust_op_proto
CMake Warning (dev) at /usr/local/Ascend/cann-8.5.0/compiler/tikcpp/ascendc_kern

---
### 跑通所有测试用例
该部分不需要修改，直接执行即可。

In [20]:
!bash src/09.01_testcase/testcase_1/run.sh

generated input.bin and golden.bin, shape=(8, 16, 64), dtype=float32
write output to ./output.bin success
TEST PASSED!


In [21]:
!bash src/09.01_testcase/testcase_2/run.sh

generated input.bin and golden.bin, shape=(8, 16, 1743), dtype=float32
write output to ./output.bin success
TEST PASSED!


In [22]:
!bash src/09.01_testcase/testcase_3/run.sh

generated input.bin and golden.bin, shape=(4, 2028), dtype=float16
write output to ./output.bin success
TEST PASSED!


In [23]:
!bash src/09.01_testcase/testcase_4/run.sh

generated input.bin and golden.bin, shape=(32, 1001, 7763), dtype=float16
write output to ./output.bin success
TEST PASSED!


In [24]:
!bash src/09.01_testcase/testcase_5/run.sh

generated input.bin and golden.bin, shape=(1, 1024), dtype=bfloat16
write output to ./output.bin success
TEST PASSED!


In [25]:
!bash src/09.01_testcase/testcase_6/run.sh

generated input.bin and golden.bin, shape=(64, 64, 1024), dtype=bfloat16
write output to ./output.bin success
TEST PASSED!
